In [6]:
import numpy as np
from bayes_opt import BayesianOptimization

n = 5
mu = np.array([
    8.6033358901938017e-01, 3.4256184594817283e+00, 6.4372981791719468e+00,
    9.5293344053619631e+00, 1.2645287223856643e+01, 1.5771284874815882e+01,
    1.8902409956860023e+01, 2.2036496727938566e+01, 2.5172446326646664e+01,
    2.8309642854452012e+01, 3.1447714637546234e+01, 3.4586424215288922e+01,
    3.7725612827776501e+01, 4.0865170330488070e+01, 4.4005017920830845e+01,
    4.7145097736761031e+01, 5.0285366337773652e+01, 5.3425790477394663e+01,
    5.6566344279821521e+01, 5.9707007305335459e+01, 6.2847763194454451e+01,
    6.5988598698490392e+01, 6.9129502973895256e+01, 7.2270467060308960e+01,
    7.5411483488848148e+01, 7.8552545984242926e+01, 8.1693649235601683e+01,
    8.4834788718042290e+01, 8.7975960552493220e+01, 9.1117161394464745e+01
])
x = np.ones(5)  /np.sqrt(5)
def rho(x):
    """Calculate the rho function for given x and mu values."""
    n = len(x)
    result = np.zeros(30)
    for j in range(30):
        exp_term = np.exp(-mu[j]**2 * np.sum(x**2))
        sum_term = sum(2 * (-1)**(ii-1) * np.exp(-mu[j]**2 * np.sum(x[ii-1:]**2)) for ii in range(2, n+1))
        result[j] = -(exp_term + sum_term + (-1)**n) / mu[j]**2
    return result

def A(mu_val):
    """Calculate the A function for a given mu value."""
    return 2 * np.sin(mu_val) / (mu_val + np.sin(mu_val) * np.cos(mu_val))

A_values= A(mu)

#BAYESIAN
def black_box(mu0, mu1, x):

    A_values = A(mu)
    mu_modified = mu.copy()
    mu_modified[0] = mu0
    mu_modified[1] = mu1

    rho_values = rho(x)

    term1 = sum(mu_modified[i]**2 * mu_modified[j]**2 * A_values[i] * A_values[j] * rho_values[i] * rho_values[j] *
                (np.sin(mu_modified[i]+mu_modified[j])/(mu_modified[i]+mu_modified[j]) + np.sin(mu_modified[i]-mu_modified[j])/(mu_modified[i]-mu_modified[j]))
                for i in range(30) for j in range(i+1, 30))

    term2 = sum(mu_modified[j]**4 * A_values[j]**2 * rho_values[j]**2 *
                (np.sin(2*mu_modified[j])/(2*mu_modified[j]) + 1)/2 for j in range(30))

    term3 = sum(mu_modified[j]**2 * A_values[j] * rho_values[j] *
                (2*np.sin(mu_modified[j])/mu_modified[j]**3 - 2*np.cos(mu_modified[j])/mu_modified[j]**2)
                for j in range(30))

    return (term1 + term2 - term3 + 2/15 - 0.0001)

pbounds = {
    'mu0': (mu[0] * -0.5, mu[0] * 1.2),
    'mu1': (mu[1] * -0.5, mu[1] * 1.2)
}  # Optimize only selected indices

def bo_wrapper(mu0, mu1):
    return black_box(mu0, mu1, x)

optimizer = BayesianOptimization(f=bo_wrapper, pbounds=pbounds, random_state=42)

# Run Bayesian Optimization
optimizer.maximize(init_points=5, n_iter=10)

# Print best result
print(optimizer.max["target"])

|   iter    |  target   |    mu0    |    mu1    |
-------------------------------------------------
| 1         | 0.1619    | 0.1176    | 3.824     |
| 2         | 0.1128    | 0.6404    | 1.774     |
| 3         | 0.1375    | -0.202    | -0.8044   |
| 4         | 0.1526    | -0.3452   | 3.331     |
| 5         | 0.1397    | 0.449     | 2.411     |
| 6         | 0.1245    | 0.657     | 3.448     |
| 7         | 0.08288   | 0.7815    | 0.03587   |
| 8         | 0.1508    | -0.3476   | 3.969     |
| 9         | 0.1252    | -0.4219   | 0.988     |
| 10        | 0.1431    | -0.4234   | 2.545     |
| 11        | 0.1346    | -0.4152   | -1.692    |
| 12        | 0.1456    | 0.4096    | 4.11      |
| 13        | 0.06637   | 1.015     | -1.692    |
| 14        | 0.1188    | -0.428    | -0.01869  |
| 15        | 0.1644    | 0.05789   | 3.015     |
0.16435247664896607
